# HOUSE-3 · 1 — Three parties, one house, and the number that was never a lift

**The situation.** A measurement house runs incrementality work for three advertisers.
Each has a weekly panel — forty markets, fifty-two weeks, spend by channel, and a KPI
that counts *conversions* — and each has a fitted media-mix model that reports what every
channel contributed last year. Each of the three has, at some point, quoted that
contribution to its board as *the lift*.

It is not the lift. The model was fitted to a year in which every one of these
advertisers bought more media when demand was up, and nothing in the panel says which
weeks those were. The model attributes the demand to the channel. That is the ordinary
way a measurement house reports a wrong number, and this notebook is the part of the
year that comes before any experiment: writing down what each party is actually asking,
showing why the observational answer cannot be trusted, and setting up the house so that
the year's experiments are filed somewhere they can be compared.

Marketing words appear in this case study — channel, spend, KPI, market, ROAS — and they
appear only through `axiom.adapters.marketing`. Every one of them is an alias for a
general object (`Channel` **is** `Treatment`, a market is the panel `Unit`, `KPI` **is**
`Outcome`), and every number is computed by the general machinery underneath. This is
the adapter earning its keep: rule 2 says the core never learns these words, and the
case study shows that it never needed to.

| | reaches into |
|---|---|
| the panels, through the adapter | `adapters.MarketingRoles`, `adapters.panel_from_marketing_frame`, `adapters.role_map`, `adapters.Channel`, `adapters.Geo`, `adapters.KPI` |
| each party's question | `adapters.contribution`, `adapters.roas`, `adapters.marginal_roas`, `surface.fit` |
| why the model's read is not the lift | `identify.CausalGraph`, `identify.identify`, `identify.adjustment_sets` |
| the house, as scopes | `io.Catalog`, `io.Program`, `build.ExperimentBuilder`, `io.DefinitionRegistry` |
| the year, as proposed | `design.Occupancy` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

import house as H
import year as Y
from axiom.display import enable, table
from _style import (
    AQUA, BLUE, CRITICAL, GOOD, MUTED, ORANGE, VIOLET, WARNING,
    annotate, caption, compare, curve_band, density, dumbbell, figure, heat, intervals,
    lines, mark_x, mark_y, points, shade, steps,
)

enable();  # every axiom result renders itself from here on

table(
    [[p.name, p.business, p.conversion.replace("_", " "), ", ".join(p.channels),
      f"{p.baseline:,.0f}", "yes" if p.order_value else "no"]
     for p in H.PARTIES.values()],
    headers=("party", "business", "a conversion is", "channels", "conversions / week, size-1 market", "reports revenue"),
    title="the three parties",
)
print(f"{H.N_MARKETS} markets in three size tiers ({', '.join(f'{t}: {H.TIERS.count(t)}' for t in ('large', 'medium', 'small'))}), "
      f"{H.N_WEEKS} weeks, spend in {H.SPEND_UNIT} per market-week")

## 1 · The year, as data

Each party hands the house a wide frame: one row per market and week, a conversions
column, one spend column per channel. `MarketingRoles` says which column plays which
role and `panel_from_marketing_frame` turns it into a `data.Panel`. The interesting line
is `role_map`: the marketing roles translate to the general `RoleMap` with nothing left
over — a channel is a `Treatment` with a currency dose, a market is a cluster `Unit`, the
KPI is an `Outcome`.

In [ ]:
from axiom.adapters import KPI, Channel, Geo, MarketingRoles, panel_from_marketing_frame, role_map
from axiom.core import Outcome, Treatment, Unit

last_year = {name: H.world(name, year=0) for name in H.PARTIES}
frames = {name: w.frame() for name, w in last_year.items()}
display(frames["northwind"].head(6))

roles = H.roles_for(H.PARTIES["northwind"])
mapped = role_map(roles)
print("Channel is Treatment:", Channel is Treatment, "| Geo is Unit:", Geo is Unit, "| KPI is Outcome:", KPI is Outcome)
print("unit column:", mapped.unit, "| time column:", mapped.time, "| outcome:", mapped.outcome[1].name,
      f"({mapped.outcome[1].dimension})")
table([[c, str(t.dimension), t.unit] for c, t in mapped.treatments.items()],
      headers=("treatment", "dimension", "unit"), title="northwind's channels, as the core sees them")

panels = {name: w.panel() for name, w in last_year.items()}
table([[name, len(p.units), len(p.periods), str(p.completeness().balanced)] for name, p in panels.items()],
      headers=("party", "markets", "weeks", "balanced"))

## 2 · Each party's question, as an estimand

"What did search contribute?", "what is our ROAS?", and "what would the next thousand
dollars return?" are three different quantities: a **contrast** (KPI at observed spend
against KPI at zero), a **ratio** (that contrast per unit of spend), and a **marginal**
(the derivative at the spend the party is actually at). The adapter names them
`contribution`, `roas` and `marginal_roas`, and every one of them is an
`estimands.Estimand` realized against a fitted surface — so each comes back with an
interval, a dimension, a unit and a ledger.

The surface is the media-mix model: Hill saturation and geometric carryover per channel,
a per-market intercept, second-order annual seasonality. `adapters.marketing_spec` builds
it from the panel; the house's `analyst_spec` only sets the prior scales to the KPI's
magnitude, because `marketing_spec` defaults every scale to one and a half-normal(1) prior
on an amplitude of ten does not fail loudly — it fits badly (note 0009 §3).

In [ ]:
from axiom.adapters import contribution, marginal_roas, roas
from axiom.core import Unsupported
from axiom.surface import fit

fits = {}
for name, panel in panels.items():
    spec = H.analyst_spec(panel, H.PARTIES[name])
    fits[name] = fit(spec, panel, backend="laplace", draws=600, seed=0)
    print(f"{name:10s} {spec.name}: {len(spec.treatments)} channels, intercept {spec.intercept!r}, "
          f"converged {fits[name].converged}")

rows = []
for name, result in fits.items():
    for channel in H.PARTIES[name].channels:
        c = contribution(result, channel, seed=0)
        rows.append([name, channel, f"{c.summary.mean:,.0f}", c.summary.interval.text(), c.unit or "", c.status])
table(rows, headers=("party", "channel", "contribution", "90% HDI", "unit", "status"),
      title="what each model says each channel contributed last year")

`roas` is a *dimensionless* return — conversions per kUSD is not one, and the adapter says
so rather than printing a number under the wrong name. Northwind also reports revenue, so
the same year can be read with a currency KPI, and there the ratio is a ROAS.

In [ ]:
refused = roas(fits["northwind"], "search", seed=0)
assert isinstance(refused, Unsupported)
print("roas on a count KPI ->", type(refused).__name__)
print("  ", refused.reason)

revenue_panel = last_year["northwind"].panel(kpi="revenue")
revenue_fit = fit(H.analyst_spec(revenue_panel, H.PARTIES["northwind"]), revenue_panel, backend="laplace", draws=600, seed=0)
money = {}
for fn in (contribution, roas, marginal_roas):
    out = fn(revenue_fit, "search", seed=0)
    money[fn.__name__] = out
table([[k, v.kind, f"{v.summary.mean:.3f}" if k != "contribution" else f"{v.summary.mean:,.0f}", v.summary.interval.text(), v.unit or "—"]
       for k, v in money.items()],
      headers=("estimand", "kind", "mean", "90% HDI", "unit"), title="northwind search, read on revenue")
print(f"\naverage return per kUSD {money['roas'].summary.mean:.2f}; return on the next kUSD {money['marginal_roas'].summary.mean:.2f}")

## 3 · Why the model's read is not the lift

Write the graph down. Spend is chosen by a planner who can see demand; conversions
respond to demand directly. The seasonal part of demand is in the model — the Fourier
terms — but the market-level, week-to-week part is not in any column, and the path
`search <- demand -> conversions` stays open whatever is conditioned on.

In [ ]:
from axiom.identify import CausalGraph, adjustment_sets, identify, requires_unmeasured

observational = CausalGraph.from_edges(
    "season -> search, season -> conversions, demand -> search, demand -> conversions, "
    "search -> conversions, market -> search, market -> conversions",
    unmeasured=["demand"], name="what_the_panel_knows",
)
verdict = identify(observational, "search", "conversions")
print("status:", verdict.status, "| route:", verdict.route)
print("admissible adjustment sets:", adjustment_sets(observational, "search", "conversions") or "none")
print("needs an unmeasured node:", requires_unmeasured(observational, "search", "conversions"))
table([[a.state, a.name, a.statement] for a in verdict.verdict.assumptions],
      headers=("state", "assumption", "what believing the model's number takes"))

experiment = CausalGraph.from_edges(
    "season -> conversions, demand -> conversions, search -> conversions, market -> conversions",
    unmeasured=["demand"], name="what_a_holdout_knows",
)
after = identify(experiment, "search", "conversions")
print("\nafter randomizing search by market:", after.status, "| adjust for", after.adjustment_set or "nothing")

The world is synthetic, so the truth is known: `World.contribution` evaluates the same
model at the truth with the channel on and off. Every model overstates every channel,
and it does so in the direction that flatters the channel — because the confounder is
demand, and demand is good news.

In [ ]:
rows, labels, said, truth = [], [], [], []
for name, result in fits.items():
    w = last_year[name]
    for channel in H.PARTIES[name].channels:
        c = contribution(result, channel, seed=0)
        true_total = float(w.contribution(channel).sum())
        rows.append([name, channel, f"{c.summary.mean:,.0f}", f"{true_total:,.0f}", f"{c.summary.mean / true_total:.2f}x",
                     "yes" if c.summary.interval.lower <= true_total <= c.summary.interval.upper else "no"])
        labels.append(f"{name} · {channel}"); said.append(c.summary.mean / true_total); truth.append(1.0)
table(rows, headers=("party", "channel", "model says", "the truth", "ratio", "truth inside the 90% HDI"),
      title="last year's contribution: the model against the world")
overstatement = float(np.mean(said))
n_outside = sum(1 for r in rows if r[-1] == "no")
fig = compare(labels, said, highlight=labels[int(np.argmax(said))],
              value_fmt="{:.2f}x", title="Seven channels, seven overstatements",
              subtitle="the model's contribution as a multiple of the true one, last year",
              x_title="model / truth")
mark_x(fig, 1.0, text="exactly right")
caption(fig, f"{n_outside} of {len(rows)} intervals miss their own truth and the average overstatement is {overstatement:.2f}x. "
             "The intervals are honest about the sampling noise and silent about the confounder, "
             "because a confounder is not noise: it is a path in the graph, and the graph said so before the fit.")

That is the argument for the year's experiments. It is also the argument for the rest of
this notebook: a geo experiment answers the question the model cannot, and a house
running eight of them for three parties has to keep track of whose answer is whose,
what each was committed to measuring, and what "a conversion" meant to the party that
filed it.

## 4 · The house, as scopes

An `io.Program` is a party and a line of work, and every artifact the house files
belongs to exactly one. An `ExperimentBuilder` per party is the front door: it plans,
commits, reads and files in that party's scope and cannot write into another's.

In [ ]:
catalog, programs, builders = Y.open_house()
table([[p.scope, p.started, p.description] for p in catalog.programs()],
      headers=("scope", "started", "line of work"), title="registered programmes")
print("the catalog lives at", catalog.root)
print("builders:", {name: repr(b) for name, b in builders.items()})

### What does this party mean by a conversion?

Every party's KPI column is called `conversions` and the three do not mean the same
thing by it. Northwind counts a paid order. Acme, in January, counts a trial start. Globex
has not said. The `DefinitionRegistry` versions whatever `Spec` a party says a term
means, in that party's scope, and `consensus` refuses to read silence as agreement: a
party that never registered the term has not agreed with anybody, so the verdict is
`unverified`, not `identified`. Once globex registers, the three *disagree*, and that
is `blocked`; only acme's change in week twelve brings them into agreement.

In [ ]:
from axiom.io import DefinitionRegistry

definitions: DefinitionRegistry = builders["northwind"].definitions
Y.register_definitions(builders, through_week=0)
january = definitions.consensus("conversion", list(programs.values()))
print(january.summary())
print(" verdict:", january.verdict().status, "—", january.verdict().reason[:110])

Y.register_definitions(builders, through_week=6)
february = definitions.consensus("conversion", list(programs.values()))
print("\nafter globex registers (week 6):", february.verdict().status)
print(" ", february.verdict().reason[:160])

Y.register_definitions(builders, through_week=12)
march = definitions.consensus("conversion", list(programs.values()))
print("\nafter acme moves to paid subscriptions (week 12):", march.verdict().status)
for change in definitions.changes(programs["acme"], "conversion"):
    print(f"  acme, {change.at}: v{change.from_version} -> v{change.to_version};",
          {k: v for k, v in change.changed.items()})

The change is recorded as a diff with a date, not a flag. Notebook 5 will ask, for every
record that enters a pool, whether the party's definition at the time was the reference's
— and acme's experiments all read after week 12, which is the only reason they can be
pooled with northwind's at all.

## 5 · The year, as proposed

Each party proposed its own experiments without looking at the others'. An
`Occupancy` is what one experiment takes up on the calendar — the markets it touches,
both arms, its weeks, and the lever it moves — and it is what notebook 2 checks for
collisions before anything is committed.

In [ ]:
calendar = [e for e in H.CALENDAR]
table([[e.id, e.party, e.channel, e.kind, f"{e.start}-{e.stop}", e.n_treated, "yes" if e.randomized else "no", e.note or "—"]
       for e in calendar],
      headers=("id", "party", "lever", "kind", "weeks", "treated markets", "randomized", "note"),
      title="the calendar as the three parties proposed it")

occupancies = {e.id: H.occupancy(e, H.MARKETS if e.kind != "rollout" else H.MARKETS[:e.n_treated]) for e in calendar}
print("an Occupancy:", occupancies["NW-01"].experiment, "| units", len(occupancies["NW-01"].units),
      "| window", occupancies["NW-01"].window, "| moves", occupancies["NW-01"].treatments)

fig = figure(title="Ten proposals on one calendar", subtitle="the weeks each experiment occupies, by party",
             x_title="week of the year", y_title="", height=380)
order = [e.id for e in calendar][::-1]
for i, e in enumerate(calendar):
    color = H.PARTY_COLOR[e.party]
    fig.add_scatter(x=[e.start, e.stop], y=[e.id, e.id], mode="lines",
                    line={"color": color, "width": 12 if e.randomized else 6, "dash": "solid" if e.randomized else "dot"},
                    name=e.party, showlegend=e.id.endswith("01"), hovertemplate=f"{e.id}: {e.channel} weeks {e.start}-{e.stop}<extra></extra>")
fig.update_yaxes(categoryorder="array", categoryarray=order, showgrid=False)
fig.update_xaxes(range=[0, 52], showgrid=True)
caption(fig, "Three pairs overlap in time. Whether an overlap matters depends on whether the two move the same lever "
             "on the same markets — the dotted bar is not an experiment at all, and the calendar has to know that too.")

## What notebook 1 established

1. **The marketing vocabulary is a translation layer.** `Channel is Treatment`,
   `Geo is Unit`, `KPI is Outcome`; the role map has nothing left over, and every
   quantity below is realized by `estimands.realize` against a `surface.fit`.
2. **Contribution, ROAS and marginal ROAS are three estimands**, each with an interval
   and a unit; a ROAS on a count KPI is a typed `Unsupported`, not a number under the
   wrong name.
3. **The model's contribution is not the lift.** `identify` returns `downgraded` on the
   `backdoor_unmeasured` route because demand drives both spend and conversions, and the
   world confirms it: every channel is overstated, five of seven by more than their own
   interval allows.
4. **The house is a catalog of scopes.** Three programmes, three builders, and a
   definition registry that records that acme changed what a conversion means in week
   twelve — and that globex had not said until week six.
5. **The calendar is a set of occupancies**, three of which overlap. Notebook 2 decides
   which of them may.